In [1]:
import pandas as pd
from scipy.io import loadmat
import os
import numpy as np
import matplotlib.pyplot as plt

In [2]:
raw_data_path = "../data/raw/"

In [3]:
data_filename = "B0005"
mat_data = loadmat(raw_data_path + data_filename + ".mat")
print(mat_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'B0005'])


In [4]:
print(mat_data["__header__"])
print(mat_data["__version__"])
print(mat_data["__globals__"])

b'MATLAB 5.0 MAT-file, Platform: PCWIN, Created on: Wed Oct 01 15:50:14 2008'
1.0
[]


In [5]:
cell_data = mat_data["B0005"]
total_cycles = len(cell_data[0,0]['cycle'][0])
print(total_cycles)

616


In [6]:
data = cell_data[0,0]['cycle'][0,0]
print(data.dtype)
print(f"Event: {data['type']}")
print(f"Ambient Temperature: {data['ambient_temperature']}")
print(data['data'][0,0].dtype)

[('type', 'O'), ('ambient_temperature', 'O'), ('time', 'O'), ('data', 'O')]
Event: ['charge']
Ambient Temperature: [[24]]
[('Voltage_measured', 'O'), ('Current_measured', 'O'), ('Temperature_measured', 'O'), ('Current_charge', 'O'), ('Voltage_charge', 'O'), ('Time', 'O')]


In [37]:
charge_fldr = "../data/processed/" + data_filename + "/charge/"
discharge_fldr = "../data/processed/" + data_filename + "/discharge/"

In [14]:
cycle_data = data['data'][0,0]
print(cycle_data['Time'])

[[0.000000e+00 2.532000e+00 5.500000e+00 8.344000e+00 1.112500e+01
  1.389100e+01 1.667200e+01 1.950000e+01 2.228200e+01 2.506300e+01
  2.782800e+01 3.064100e+01 3.345300e+01 3.621900e+01 3.973500e+01
  4.257800e+01 4.543800e+01 4.829700e+01 5.118800e+01 5.404700e+01
  5.692200e+01 5.979700e+01 6.268800e+01 6.565700e+01 6.854700e+01
  7.145300e+01 7.434400e+01 7.723500e+01 8.018800e+01 8.317200e+01
  8.609400e+01 8.901600e+01 9.192200e+01 9.490700e+01 9.784400e+01
  1.007660e+02 1.037500e+02 1.067030e+02 1.096410e+02 1.126410e+02
  1.155940e+02 1.185470e+02 1.215320e+02 1.245000e+02 1.275320e+02
  1.305160e+02 1.335630e+02 1.365630e+02 1.395630e+02 1.425630e+02
  1.455940e+02 1.485780e+02 1.516100e+02 1.546720e+02 1.577350e+02
  1.607820e+02 1.638440e+02 1.669220e+02 1.700320e+02 1.731100e+02
  1.761880e+02 1.792660e+02 1.824220e+02 1.855630e+02 1.887030e+02
  1.918600e+02 1.950000e+02 1.981880e+02 2.013600e+02 2.045000e+02
  2.076570e+02 2.108130e+02 2.139690e+02 2.171100e+02 2.202970

In [46]:
import shutil

if os.path.exists(charge_fldr) == True:
    shutil.rmtree(charge_fldr)
    os.makedirs(charge_fldr)
    print(f"Created folder: {charge_fldr}")
else:
    os.makedirs(charge_fldr)
    print(f"Created folder: {charge_fldr}")


if os.path.exists(discharge_fldr) == True:
    shutil.rmtree(discharge_fldr)
    os.makedirs(discharge_fldr)
    print(f"Created folder: {discharge_fldr}")
else:
    os.makedirs(discharge_fldr)
    print(f"Created folder: {discharge_fldr}")

charge_count = 0
discharge_count = 0
impedance_count = 0
for i in range(total_cycles):
    raw_data = cell_data[0,0]['cycle'][0,i]
    event_type = raw_data['type'][0]
    ambient_temp = raw_data['ambient_temperature'][0,0]
    cycle_data = raw_data['data'][0,0]

    if event_type == "discharge":
        filename = discharge_fldr + f"dchgCycle_{discharge_count}.csv"
        df_cycle = pd.DataFrame()
        df_cycle["time"] = cycle_data['Time'][0]
        df_cycle["current"] = cycle_data['Current_measured'][0]
        df_cycle["voltage"] = cycle_data['Voltage_measured'][0]
        df_cycle["temperature"] = cycle_data['Temperature_measured'][0]
        # df_cycle["capacity"] = cycle_data['Capacity'][0]
        # print(df_cycle.head())
        df_cycle.to_csv(filename, index=False)
        discharge_count += 1

    if event_type == "charge":
        filename = charge_fldr + f"chgCycle_{charge_count}.csv"
        df_cycle = pd.DataFrame()
        df_cycle["time"] = cycle_data['Time'][0]
        df_cycle["current"] = cycle_data['Current_measured'][0]
        df_cycle["voltage"] = cycle_data['Voltage_measured'][0]
        df_cycle["temperature"] = cycle_data['Temperature_measured'][0]
        # df_cycle["capacity"] = cycle_data['Capacity'][0]
        # print(df_cycle.head())
        df_cycle.to_csv(filename, index=False)
        charge_count += 1
    
    if event_type == "impedance":
        impedance_count += 1
    

print(f"Discharge Files: {discharge_count}")
print(f"Charge Files: {charge_count}")
print(f"Impedance Files: {impedance_count}")

Created folder: ../data/processed/B0005/charge/
Created folder: ../data/processed/B0005/discharge/
Discharge Files: 168
Charge Files: 170
Impedance Files: 278
